# AI とチャットする（LINE のような吹き出しで）

`ui.chat()` は会話を吹き出しで並べる部品です。自分の言葉は右、AI の言葉は左に出ます。
これに `ai` を組み合わせると、1つのセルで会話ができます。

**この教材は2部に分かれています。環境によって、できることが違うためです。**

| | Google Colab | PyHiroba |
|---|---|---|
| 前半：コードから話しかける | 動きます | 動きます |
| 後半：入力欄に打ち込んで話す | 動きます | **まだ動きません** |

後半が PyHiroba で動かないのは、画面に打った文字を Python に戻す道が本体側にまだ無いためです
（`ui.form()` は表示されますが、ボタンを押しても何も起きません）。詳しくは
[`docs/PYHIROBA_FORMS.md`](../docs/PYHIROBA_FORMS.md) にあります。

**両方で使いたい教材は、前半だけで作ってください。**

In [ ]:
%pip install -q "library-hiroba[ai]"
# PyHiroba では、このセルの実行は不要です

In [ ]:
from library_hiroba import ai, ui

# "auto" は、この環境で実用になるもののうち、いちばん良いモデルを選びます
print(await ai.load("auto"))

## 会話を覚えておく

`ai.ask()` が受け取るのは**1回分の文章だけ**です。前に何を話したかは覚えていません。
そのまま使うと、「その高さは？」と聞いても何の話か分からず、答えがちぐはぐになります。

そこで、これまでのやりとりを自分で覚えておき、質問に添えて渡します。
次の `Talk` がそれをまとめて引き受けます。

In [ ]:
class Talk:
    """AI との会話を覚えておき、吹き出しにして返す。"""

    def __init__(self, keep=4, max_tokens=96, names=None):
        self.history = []
        self.keep = keep              # 覚えておく往復の数
        self.max_tokens = max_tokens  # 1回の答えの長さ
        self.names = names or {"user": "あなた", "assistant": "AI"}

    def _prompt(self, question):
        """直前のやりとりを添えた、渡す用の文章を作る。"""
        recent = self.history[-self.keep * 2:]
        lines = ["これまでの会話です。AI として、最後の質問に日本語で短く答えてください。", ""]
        for message in recent:
            who = "あなた" if message["role"] == "user" else "AI"
            lines.append(f"{who}: {message['content']}")
        lines.append(f"あなた: {question}")
        lines.append("AI:")
        return "\n".join(lines)

    def _clean(self, text):
        """小さなモデルは、答えたあとに自分で会話の続きを書き足すことがある。"""
        for marker in ("あなた:", "あなた：", "\nAI:"):
            if marker in text:
                text = text.split(marker)[0]
        return text.strip()

    def _view(self, extra=None):
        messages = self.history if extra is None else [*self.history, extra]
        return ui.chat(messages, names=self.names)

    async def __call__(self, question):
        """1往復して、会話ぜんぶを吹き出しで返す。"""
        prompt = self._prompt(question)
        self.history.append({"role": "user", "content": question})
        answer = self._clean(await ai.ask(prompt, max_tokens=self.max_tokens))
        self.history.append({"role": "assistant", "content": answer})
        return self._view()

    async def stream(self, question):
        """同じことを、書けたところから少しずつ返す。"""
        prompt = self._prompt(question)
        self.history.append({"role": "user", "content": question})
        text = ""
        async for chunk in ai.stream(prompt, max_tokens=self.max_tokens):
            text += chunk
            partial = self._clean(text)
            if partial:
                yield self._view({"role": "assistant", "content": partial})
        self.history.append({"role": "assistant", "content": self._clean(text)})
        yield self._view()


talk = Talk()

## 前半：コードから話しかける（Colab・PyHiroba の両方）

`await talk("...")` をセルの最後に置くと、そこまでの会話が吹き出しで出ます。
セルを増やすたびに、話が続きます。

In [ ]:
await talk("日本で一番高い山は？")

In [ ]:
# 「その」が何を指すか、前のやりとりから分かります
await talk("その高さは？")

会話をやり直したいときは、新しく作り直します。

```python
talk = Talk()
```

`Talk(keep=1)` にすると直前の1往復だけを覚えます。小さなモデルは長い文章が苦手なので、
話がかみ合わなくなってきたら減らしてみてください。

## 後半：入力欄に打ち込んで話す（いまは Colab・Jupyter だけ）

`ui.form()` を重ねると、入力欄から話しかけられます。`Talk.stream()` を使うので、
答えが書けたところから少しずつ吹き出しに出ます。

**このセルは PyHiroba では動きません**（フォームは出ますが、送信しても何も起きません）。

In [ ]:
talk = Talk()  # ここまでの会話は引き継がず、新しく始めます


async def send(message):
    async for view in talk.stream(message):
        yield view


ui.form(
    send,
    ui.field("message", label="", placeholder="メッセージを入力"),
    submit_label="送信",
    clear_on_submit=True,
)

---

確認ポイント:

- `await ai.load("auto")` が「準備ができました」と表示する
- `await talk("日本で一番高い山は？")` で吹き出しが2つ出る（あなた／AI）
- 続けて `await talk("その高さは？")` を実行すると、山の話として答える
- （Colab のみ）入力欄に打って送信すると、答えが少しずつ書き足されていく

小さなモデルなので、答えが事実と違うことがあります。教材では「AI の答えを確かめる」
題材として使うのが向いています。

表示名は変えられます。

```python
talk = Talk(names={"user": "生徒", "assistant": "先生"})
```